# Project Description

This project evaluates the performance of test and control groups across key funnel metrics: impressions, clicks, purchases, and revenue.  





# Contents


1. [Data Load SQL](#data-load-sql)
2. [AA Test: One Sample T-test](#aa-test-one-sample-t-test)
3. [Initial Two Sample T-test](#initial-two-sample-t-test)
4. [Create Table for Tableau](#create-table-for-tableau)
5. [Segmented A/B Test Dashboard](#segmented-ab-test-dashboard)
6. [Segment-Level Insight](#segment-level-insight)
7. [Business Recommendation](#business-recommendation)

# Setting

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import os
os.chdir('gdrive/MyDrive/AB_testing/data')
!ls

aa-example.csv	duckdb.db  user-event.csv  user-metadata.csv  user-variant.csv


In [ ]:
!pip install --quiet duckdb
!pip install --quiet jupysql
!pip install --quiet duckdb-engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.0/264.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00


In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#%load_ext sql
%reload_ext sql

In [ ]:
%sql duckdb:///duckdb.db

Connecting to 'duckdb:///duckdb.db'

<a name="data-load-sql"></a>
# **Data Load (SQL)**
* Load the required files from cloud data storage
* Database Setup: Creating Schemas and Tables for analysis

In [ ]:
%%sql

create schema if not exists raw_data;
create schema if not exists analytics;

Running query in 'duckdb:///duckdb.db'

Count


In [ ]:
%%sql

create table if not exists raw_data.user_event(
  user_id int,
  datestamp timestamp,
  item_id int,
  clicked int,
  purchased int,
  paidamount int
);

Running query in 'duckdb:///duckdb.db'

Count


In [ ]:
%%sql

create table if not exists raw_data.user_variant (
  user_id int,
  variant_id varchar(32)
);

Running query in 'duckdb:///duckdb.db'

Count


In [ ]:
%%sql

create table if not exists raw_data.user_metadata(
  user_id int,
  age varchar(16),
  country varchar(32)
);

Running query in 'duckdb:///duckdb.db'

Count


In [ ]:
%%sql

create table if not exists raw_data.aa_example(
  user_id int,
  date date,
  job_position_id int,
  clicked int,
  checkedout int,
  applied int
);

Running query in 'duckdb:///duckdb.db'

Count


In [ ]:
%%sql

SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema IN ('raw_data', 'analytics');

Running query in 'duckdb:///duckdb.db'

table_schema,table_name
analytics,analytics_variant_user_daily
analytics,variant_daily_session
analytics,variant_daily_sessions
raw_data,aa_example
raw_data,user_event
raw_data,user_metadata
raw_data,user_variant


In [ ]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [ ]:
!wget https://s3-geospatial.s3.us-west-2.amazonaws.com/user-metadata.csv
!wget https://s3-geospatial.s3.us-west-2.amazonaws.com/user-event.csv
!wget https://s3-geospatial.s3.us-west-2.amazonaws.com/user-variant.csv
!wget https://s3-geospatial.s3.us-west-2.amazonaws.com/aa-example.csv

In [ ]:
%%sql

INSERT INTO raw_data.user_metadata SELECT * FROM 'user-metadata.csv';
INSERT INTO raw_data.user_variant SELECT * FROM 'user-variant.csv';
INSERT INTO raw_data.user_event SELECT * FROM 'user-event.csv';
INSERT INTO raw_data.aa_example SELECT * FROM 'aa-example.csv';

,Success


In [ ]:
%%sql

select * from raw_data.user_event limit 10;

,user_id,datestamp,item_id,clicked,purchased,paidamount
0,91139,2019-01-13,34,0,0,0
1,91139,2019-01-13,77,0,0,0
2,91139,2019-01-13,17,0,0,0
3,40135,2019-01-14,54,1,0,0
4,40135,2019-01-14,2,0,0,0
5,40135,2019-01-14,96,0,0,0
6,40135,2019-01-14,23,0,0,0
7,40135,2019-01-14,12,1,0,0
8,40135,2019-01-14,14,0,0,0
9,40135,2019-01-14,68,0,0,0


In [ ]:
%%sql
select * from raw_data.user_variant limit 10;

,user_id,variant_id
0,0,test
1,1,test
2,2,control
3,3,test
4,4,control
5,5,test
6,6,control
7,7,test
8,8,control
9,9,control


In [ ]:
%%sql
select * from raw_data.user_metadata limit 10;

,user_id,age,gender
0,0,50-up,female
1,1,50-up,female
2,2,0-19,female
3,3,50-up,female
4,4,0-19,male
5,5,0-19,female
6,6,0-19,undefined
7,7,50-up,female
8,8,50-up,male
9,9,0-19,male


In [ ]:
%%sql
select * from raw_data.aa_example limit 10;

,user_id,date,job_position_id,clicked,checkedout,applied
0,6810,2022-01-03,4891,1,1,0
1,117686,2022-01-03,8882,1,1,1
2,117686,2022-01-03,9006,1,1,1
3,316741,2022-01-03,8331,1,1,1
4,380002,2022-01-03,2385,1,1,1
5,117686,2022-01-03,8683,1,1,1
6,374410,2022-01-03,9014,1,1,1
7,285735,2022-01-03,1458,1,1,0
8,380002,2022-01-03,7088,1,1,1
9,395070,2022-01-03,5482,1,1,1


### create variant_daily_session table under analytics schema

* Interests: Total Impression, Total Click, Total Purchase, Total Revenue

In [ ]:
%%sql
select * from raw_data.user_variant limit 10;

,user_id,variant_id
0,0,test
1,1,test
2,2,control
3,3,test
4,4,control
5,5,test
6,6,control
7,7,test
8,8,control
9,9,control


In [ ]:
%%sql

create table analytics.variant_daily_sessions as
select
  uv.variant_id,
  ue.user_id,
  ue.datestamp,
  count(distinct ue.item_id) as num_of_items,
  sum(ue.clicked) as num_of_clicks,
  sum(ue.purchased) as num_of_purchases,
  sum(ue.paidamount) as revenue
from raw_data.user_variant uv
join raw_data.user_event ue
on uv.user_id = ue.user_id
group by 1,2,3;


In [ ]:
%%sql

select *
from analytics.variant_daily_sessions
limit 100;

,variant_id,user_id,datestamp,num_of_items,num_of_clicks,num_of_purchases,revenue
0,control,40135,2019-01-12,18,8.0,2.0,68.0
1,test,89370,2019-01-16,2,0.0,0.0,0.0
2,test,89370,2019-01-15,2,0.0,0.0,0.0
3,test,89371,2019-01-13,7,2.0,0.0,0.0
4,control,89376,2019-01-13,4,1.0,1.0,11.0
...,...,...,...,...,...,...,...
95,control,46863,2019-01-11,2,0.0,0.0,0.0
96,test,46862,2019-01-16,25,5.0,0.0,0.0
97,test,46862,2019-01-13,15,5.0,1.0,23.0
98,test,46862,2019-01-14,3,1.0,0.0,0.0


<a name="business-recommendation"></a>
# **Business Recommendation**

* Therefore, instead of rolling out the test variant to all users immediately, a limited rollout or further validation could be considered for the 0–19 female segment, where positive effects were observed.

* In contrast, for the 20–49 female segment, the causes of the decrease in Purchase and Revenue should be investigated further, and applying the same variant to this segment should be postponed.